# `sc_llm_x402` Buyer Notebook — real end-to-end chat payment (Deno/TypeScript)

Drives the **real, locally-running** `sc_llm_x402.ts` batch-settlement chat handler as a buyer would —
the missing real-server verification for Phase B, and the runnable blueprint
`website/hooks/useX402Chat.ts` was built from. Unlike
`x402_facilitator/notebooks/x402_batch_settlement_buyer.ipynb` (which hits the facilitator's raw
`/verify`+`/settle` endpoints directly with hand-built payment requirements), this notebook only ever
talks to the **resource server** over plain HTTP via `wrapFetchWithPayment` — exactly what a real client
(browser or otherwise) does. The server decides and advertises everything else.

**Terminology — x402 SDK role → this repo's package:**

| SDK role | Repo package | Role in this repo |
|---|---|---|
| **Client** (buyer / payer) | `website/` *(this notebook stands in for it)* | Browser wallet: signs the deposit + per-message vouchers |
| **Server** (seller / merchant) | `scw_js/` (`sc_llm_x402.ts`) | Verifies vouchers, serves the LLM response, commits the charge |
| **Facilitator** | `x402_facilitator/` | Neither buyer nor seller — executes on-chain deposit/claim/settle |

## ⚠️ Prerequisites

1. **Deno Jupyter kernel** — `deno jupyter --install` (same global kernel as the sibling facilitator notebooks).
2. **`scw_js/.env`** needs:
   - `TEST_WALLET_PRIVATE_KEY` — the buyer wallet, funded with USDC on whichever network you select
     below (Base Sepolia: https://faucet.circle.com/; Base Mainnet: a real, funded wallet). Purely a
     notebook-testing convenience — no scw_js production code reads this key.
   - `NFT_WALLET_PUBLIC_KEY` — the receiver address (the server reads the same file).
   - `RECEIVER_AUTHORIZER_PRIVATE_KEY` — required for `createLLMResourceServer()` to construct at all.
     A pure off-chain signer, no funding needed. See `assistent_plan.md` §Backlog F for why this isn't
     delegated to the facilitator (yet).
   - `MISTRAL_API_KEY` — required once `USE_MAINNET` is armed below (the **server**, not this notebook,
     reads it). Real Mistral completions are billed to this key — see `llm_service.ts`'s `LLM_PROVIDERS`.
   - `SCW_ACCESS_KEY` / `SCW_SECRET_KEY` — the server's `S3ChannelStorage` writes real objects under
     `channels/` in the production bucket when this runs (private ACL, harmless, but real).
3. **Start the local server** (separate terminal): `cd scw_js && npm run dev:llmx402` — listens on `:8085`,
   and internally talks to the **real deployed facilitator** (`https://facilitator.fretchen.eu` by default)
   unless `FACILITATOR_URL` is overridden. This means the first cell that opens a channel submits a
   **real on-chain transaction on the selected network**.
4. **Or skip step 3 entirely** and set `USE_DEPLOYED = true` in the network-selection cell below to hit
   the real deployed `llmx402` Scaleway function instead of a local server — same S3 channel storage
   bucket and same facilitator either way, so results are directly comparable. The deployed function
   scales to zero when idle, so the very first request after a while may take several extra seconds
   (cold start) before the usual settlement timing shown in this notebook's past runs.

In [1]:
// Setup: imports + env
import { load } from "https://deno.land/std@0.224.0/dotenv/mod.ts";
import { privateKeyToAccount } from "npm:viem@2/accounts";

// scw_js's own .env, one level up — the single source of truth (buyer key, receiver
// address, and the server's own secrets all live here; the server reads the same file).
const env = await load({ envPath: "../.env", examplePath: null, export: true });

const PRIVATE_KEY = env.TEST_WALLET_PRIVATE_KEY;
if (!PRIVATE_KEY) {
    throw new Error("TEST_WALLET_PRIVATE_KEY missing from scw_js/.env — add a key funded with Base Sepolia USDC.");
}
const account = privateKeyToAccount(`0x${PRIVATE_KEY.replace(/^0x/, "")}`);

console.log("🚀 sc_llm_x402 buyer notebook");
console.log(`   Buyer (payer): ${account.address}`);
console.log(`   Receiver     : ${env.NFT_WALLET_PUBLIC_KEY ?? "(not set — server will 500)"}`);

🚀 sc_llm_x402 buyer notebook
   Buyer (payer): 0x553179556FC2A39e535D65b921e01fA995E79101
   Receiver     : 0xAAEBC1441323B8ad6Bdf6793A8428166b510239C


## Network selection

`getBatchSettlementNetworks()` on the server side (`x402_server.ts`) only allows **Base** —
`["eip155:8453", "eip155:84532"]`. Optimism is excluded entirely (a separate, pre-existing
`@x402/evm` `DEFAULT_STABLECOINS` gap, not a spike limitation — filed upstream, see
`x402_facilitator/upstream/`).

| Network | CAIP-2 | Real Mistral call? |
|---|---|---|
| Base Sepolia (testnet) | `eip155:84532` | No — the server always mocks (`isTestnet`), regardless of what the client sends |
| Base Mainnet | `eip155:8453` | **Yes** — real USDC settlement + a real, billed Mistral completion |

**One flag decides everything**: `USE_MAINNET` below selects the network — and since the server mocks
every testnet request unconditionally, choosing Mistral *is* choosing mainnet. There's no separate
confirmation step; flipping this one flag is the deliberate action.


In [ ]:
import { base, baseSepolia } from "npm:viem@2/chains";

const USE_MAINNET = false;  // ⚠️ true = REAL MONEY: mainnet USDC settlement + a real, billed Mistral call.

const NETWORK_CONFIG = {
    "base-testnet": {
        caip2Network: "eip155:84532" as const, networkName: "Base Sepolia (Testnet)",
        chain: baseSepolia, usdcName: "USDC",
        usdcAddress: "0x036CbD53842c5426634e7929541eC2318f3dCF7e" as `0x${string}`,
        explorer: "https://sepolia.basescan.org", faucet: "https://faucet.circle.com/",
    },
    "base-mainnet": {
        caip2Network: "eip155:8453" as const, networkName: "Base Mainnet",
        chain: base, usdcName: "USD Coin",
        usdcAddress: "0x833589fCD6eDb6E08f4c7C32D4f71b54bdA02913" as `0x${string}`,
        explorer: "https://basescan.org", faucet: "Bridge: https://bridge.base.org",
    },
};
const config = NETWORK_CONFIG[USE_MAINNET ? "base-mainnet" : "base-testnet"];
const NETWORK = config.caip2Network;

// Toggle between the local dev server and the real deployed Scaleway function — orthogonal to
// USE_MAINNET, same code path either way, since the channel is on-chain state keyed by
// (buyer, receiver, network, asset), not by which server queried it.
// Local: `cd scw_js && npm run dev:llmx402` (see prerequisites above).
const USE_DEPLOYED = false;
const SERVICE_URL = USE_DEPLOYED
    ? "https://mypersonaljscloudivnad9dy-llmx402.functions.fnc.fr-par.scw.cloud"
    : "http://localhost:8085";

console.log(USE_MAINNET ? `🚨 REAL MONEY on ${config.networkName}` : `🧪 ${config.networkName}`);
console.log(`   ${NETWORK} • USDC ${config.usdcName} @ ${config.usdcAddress}`);
console.log(`   Service: ${SERVICE_URL}`);

🚨 REAL MONEY on Base Mainnet
   eip155:8453 • USDC USD Coin @ 0x833589fCD6eDb6E08f4c7C32D4f71b54bdA02913
   Service: http://localhost:8085


## Buyer setup — the shape `useX402Chat.ts` will need

Unlike the `exact` scheme (`registerExactEvmScheme` helper), batch-settlement has **no client-register
helper** — the scheme is constructed and registered manually, same as the facilitator's buyer spike.

For channel storage this notebook uses a tiny `WebStorageClientChannelStorage` adapter over Deno's
**global `localStorage`** — the same Web Storage API a browser exposes, and (unlike
`InMemoryClientChannelStorage`) it persists across kernel restarts, so an open channel is reused on a
re-run exactly as it must survive a page reload in production. `useX402Chat.ts` reuses this *same* class,
passing `window.localStorage` instead of Deno's global (per `assistent_plan.md` Phase C's plan).
Everything else below is copy-paste-equivalent to what the hook needs.


In [ ]:
import { x402Client, wrapFetchWithPayment, x402HTTPClient } from "npm:@x402/fetch@^2.17.0";
import { toClientEvmSigner } from "npm:@x402/evm@^2.17.0";
import {
    BatchSettlementEvmScheme,
    type ClientChannelStorage,
    type BatchSettlementClientContext,
    type BatchSettlementDepositStrategyContext,
} from "npm:@x402/evm@^2.17.0/batch-settlement/client";
import { createPublicClient, http } from "npm:viem@2";

// readContract is documented as "optional" on ClientEvmSigner (only "required for extension
// enrichment" per the SDK's own JSDoc), but batch-settlement's corrective-402 recovery
// (processCorrectivePaymentRequired, triggered on invalid_batch_settlement_evm_cumulative_amount_mismatch
// and _below_claimed) unconditionally needs it too — both recoverFromSignature and
// recoverFromOnChainState bail out immediately with `if (!deps.signer.readContract) return false`.
// Without it, a client/server cumulative-total desync (e.g. from an old channel reused across
// notebook/kernel restarts via localStorage) surfaces as a hard 402 instead of self-healing.
// toClientEvmSigner() is the SDK's own helper for composing a full signer from a plain
// account + a public client. In useX402Chat.ts: swap `account`/`publicClient` here for a wagmi
// WalletClient adapter + `usePublicClient()` (see useX402ImageGeneration.ts's signer shape).
// chain comes from `config` (network-selection cell) — mainnet/testnet selects both the
// network AND which chain object the public client reads against.
const publicClient = createPublicClient({ chain: config.chain, transport: http() });
const buyerSigner = toClientEvmSigner(
    { address: account.address, signTypedData: (a: any) => account.signTypedData(a) },
    publicClient,
);

// ClientChannelStorage backed by the Web Storage API. Channel context is all strings, so
// plain JSON round-trips. In useX402Chat.ts, construct this SAME class with window.localStorage.
class WebStorageClientChannelStorage implements ClientChannelStorage {
    constructor(private backend: Storage, private prefix = "x402-channel:") {}
    private keyFor(key: string) { return `${this.prefix}${key.toLowerCase()}`; }
    async get(key: string) {
        const raw = this.backend.getItem(this.keyFor(key));
        return raw ? (JSON.parse(raw) as BatchSettlementClientContext) : undefined;
    }
    async set(key: string, context: BatchSettlementClientContext) {
        this.backend.setItem(this.keyFor(key), JSON.stringify(context));
    }
    async delete(key: string) { this.backend.removeItem(this.keyFor(key)); }
}

// Deno exposes the global `localStorage` (persists across kernel restarts). Browser: window.localStorage.
const buyerStorage = new WebStorageClientChannelStorage(localStorage);

// Floor for channel deposits/top-ups, in USDC atomic units (6 decimals) — $0.50. Same value
// and same reasoning as useX402Chat.ts's MINIMUM_DEPOSIT_ATOMIC: the SDK's own default
// (depositMultiplier x per-message ceiling) tracks whatever the ceiling happens to be
// (currently ~$0.003/message), sizing deposits at only ~1-3 cents — enough for ~5 messages
// worst-case before another on-chain top-up (a real tx + wait) is needed. $0.50 comfortably
// covers a full session while keeping the number small on the axis that actually matters for
// this app: it's the blast radius if a delegate voucher-signer key ever leaks.
const MINIMUM_DEPOSIT_ATOMIC = 500_000n;

function depositStrategy(context: BatchSettlementDepositStrategyContext): string {
    const required = BigInt(context.minimumDepositAmount);
    return (required > MINIMUM_DEPOSIT_ATOMIC ? required : MINIMUM_DEPOSIT_ATOMIC).toString();
}

const buyerScheme = new BatchSettlementEvmScheme(buyerSigner, { storage: buyerStorage, depositStrategy });

const client = new x402Client();
client.register(NETWORK, buyerScheme);

const fetchWithPayment = wrapFetchWithPayment(fetch, client);
console.log("✅ buyer client ready, registered for", NETWORK);

## Pre-flight — buyer USDC balance vs. estimated cost

Neither the network flag above nor the warnings below change the fact that arming `USE_MAINNET`
spends real money. Check the buyer has enough USDC *before* message #1 tries to open a channel.
The deposit is governed by the `depositStrategy` set up in the previous cell — a fixed $0.50 floor
(see `MINIMUM_DEPOSIT_ATOMIC`), not the SDK's own smaller multiplier-of-ceiling default — chosen so
a full multi-message session doesn't need a mid-conversation on-chain top-up.


In [ ]:
import { formatUnits } from "npm:viem@2";

const erc20BalanceOfAbi = [{
    inputs: [{ name: "account", type: "address" }],
    name: "balanceOf",
    outputs: [{ name: "", type: "uint256" }],
    stateMutability: "view",
    type: "function",
}] as const;

// Matches the depositStrategy cell above — the actual deposit floor, not the SDK's default.
const ESTIMATED_DEPOSIT_ATOMIC = MINIMUM_DEPOSIT_ATOMIC;

const buyerUsdc = await publicClient.readContract({
    address: config.usdcAddress,
    abi: erc20BalanceOfAbi,
    functionName: "balanceOf",
    args: [account.address],
});

console.log(`💵 Buyer USDC: ${formatUnits(buyerUsdc, 6)}  (estimated deposit ≈ ${formatUnits(ESTIMATED_DEPOSIT_ATOMIC, 6)})`);
if (buyerUsdc < ESTIMATED_DEPOSIT_ATOMIC) {
    console.log(`   ⚠️ insufficient USDC on ${config.networkName} — ${config.faucet}`);
} else {
    console.log(`   ✅ enough USDC to open the channel`);
}

## Message #1 — opens the channel (real on-chain deposit tx)

The first call to `fetchWithPayment` against a resource it hasn't paid for yet: an initial bare request
gets `402`, the SDK builds a **deposit** payload (open the channel + first voucher) and signs it, the
retry carries the payment header, and the server verifies + settles (the deposit — the one real on-chain
tx per channel lifetime, confirmed in the B0 spike). Extracting the settlement receipt mirrors
`useX402ImageGeneration.ts`'s `x402HTTPClient.getPaymentSettleResponse(...)` call exactly.

> ⚠️ Sending a message performs a **real on-chain USDC settlement** on the selected network, and —
> when `USE_MAINNET` is armed — a **real, billed Mistral API call**. Both cost real money the moment
> `USE_MAINNET = true`.

> ℹ️ If a send is **interrupted** between verify and settle (kernel interrupt, network drop), the
> server leaves a short-lived per-channel lock, and the next send on that channel returns
> `invalid_batch_settlement_evm_channel_busy`. This is **intentional and self-healing** — the lock
> serializes requests on one channel and expires on its own (≤ `LLM_MAX_TIMEOUT_SECONDS`, currently
> 120s; the client SDK does not auto-recover from it). Just wait a couple of minutes and re-run. The
> website surfaces this as a friendly "wait a few seconds and try again" message
> (`useX402Chat.ts::describePaymentError`).

In [5]:
async function sendChatMessage(content: string) {
    const started = performance.now();

    // FIXED (x402_facilitator PR #545, confirmed deployed — /supported now reports the real
    // signer address, shipped in the same deploy): x402_facilitator's settlePayment() used to
    // rebuild its own SettleResult and drop the SDK's `extra` (incl. channelState.channelId),
    // so a real, correctly-settled payment's response reached the client missing channelId and
    // crashed processSettleResponse() on channelId.toLowerCase(). That crash — while it was still
    // live — could leave a channel settled server-side but never persisted client-side, which is
    // the likely origin of any lingering cumulative-amount desync on a reused channel (see the
    // `toClientEvmSigner` fix in the previous cell for why that no longer surfaces as a hard
    // failure either way). Keeping this try/catch as defense-in-depth, not because the crash is
    // expected anymore.
    //
    // No `useDummyData` field — matches the real client (useX402Chat.ts) exactly. Mock-vs-real
    // is decided entirely server-side by NETWORK (see the network-selection cell): testnet always
    // mocks, mainnet always calls the real Mistral API. There is nothing else to set here.
    let response: Response;
    try {
        response = await fetchWithPayment(SERVICE_URL, {
            method: "POST",
            headers: { "Content-Type": "application/json" },
            body: JSON.stringify({ data: { prompt: [{ role: "user", content }] } }),
        });
    } catch (err) {
        const elapsedMs = Math.round(performance.now() - started);
        console.log(`📡 fetchWithPayment THREW after ${elapsedMs}ms:`, (err as Error).message);
        console.log("   (server may have settled successfully anyway — check the server's own log)");
        return { response: null, body: null, receipt: null, elapsedMs, threw: (err as Error).message };
    }
    const elapsedMs = Math.round(performance.now() - started);

    const bodyText = await response.text();
    let body: any;
    try { body = JSON.parse(bodyText); } catch { body = bodyText; }

    console.log(`📡 status=${response.status}  elapsed=${elapsedMs}ms`);
    console.log("📨 body:", JSON.stringify(body, null, 2).slice(0, 600));

    let receipt: unknown = null;
    try {
        const httpClient = new x402HTTPClient(client);
        receipt = httpClient.getPaymentSettleResponse((name: string) => response.headers.get(name));
        console.log("🧾 settlement receipt:", JSON.stringify(receipt, null, 2));
    } catch (err) {
        console.log("🧾 no settlement receipt:", (err as Error).message);
    }
    return { response, body, receipt, elapsedMs, threw: null };
}

const msg1 = await sendChatMessage("What is the capital of France?");

📡 status=200  elapsed=12901ms
📨 body: {
  "content": "The capital of France is **Paris**.",
  "usage": {
    "prompt_tokens": 10,
    "total_tokens": 19,
    "completion_tokens": 9,
    "prompt_tokens_details": {
      "cached_tokens": 0
    }
  },
  "model": "mistral-large-latest"
}
🧾 settlement receipt: {
  "success": true,
  "payer": "0x553179556FC2A39e535D65b921e01fA995E79101",
  "transaction": "0xcf5f89bae9626c199793dd75fef91afc7b097050d674347bc1b7b851c19c10ff",
  "network": "eip155:8453",
  "extra": {
    "channelState": {
      "channelId": "0xbd47699c24d5fbd16eb316de548ab39e36931a953ee33aa30dc9b28ff5507446",
      "balance": "15000",
      "totalClaimed": "0",
      "withdrawRequestedAt": 0,
      "refundNonce": "0",
      "chargedCumulativeAmount": "18"
    },
    "chargedAmount": "18"
  }
}


## Messages #2 and #3 — same channel, the actual spike question

**What we don't know yet, and this cell exists to find out:** does `fetchWithPayment` (via
`client.createPaymentPayload()`, called internally on every `402`) automatically notice
`buyerStorage` already holds an open channel for this `(receiver, network, asset)` and sign a plain
**voucher** against it (zero on-chain tx — the actual batching payoff) — or does it try to open a
**second, wasteful deposit**? The facilitator's buyer spike only demonstrated channel reuse via the
low-level `signVoucher` helper called *manually*, never through the automatic `fetchWithPayment` path —
so this genuinely hasn't been observed yet. Whatever happens here is what `useX402Chat.ts` needs to
account for.


In [6]:
const msg2 = await sendChatMessage("And what is the capital of Germany?");
const msg3 = await sendChatMessage("And Italy?");

📡 status=200  elapsed=5345ms
📨 body: {
  "content": "The capital of Germany is **Berlin**. It has been the capital since the reunification of Germany in 1990, following the fall of the Berlin Wall in 1989. Before that, Bonn served as the capital of West Germany (Federal Republic of Germany) during the Cold War.",
  "usage": {
    "prompt_tokens": 11,
    "total_tokens": 74,
    "completion_tokens": 63,
    "prompt_tokens_details": {
      "cached_tokens": 0
    }
  },
  "model": "mistral-large-latest"
}
🧾 settlement receipt: {
  "success": true,
  "payer": "0x553179556fc2a39e535d65b921e01fa995e79101",
  "transaction": "",
  "network": "eip155:8453",
  "amount": "",
  "extra": {
    "channelState": {
      "channelId": "0xbd47699c24d5fbd16eb316de548ab39e36931a953ee33aa30dc9b28ff5507446",
      "balance": "0",
      "totalClaimed": "0",
      "withdrawRequestedAt": 0,
      "refundNonce": "0",
      "chargedCumulativeAmount": "118"
    },
    "chargedAmount": "100"
  }
}
📡 status=200  el

## Findings (observed 2026-07-16, real run against local server + real Base Sepolia facilitator)

- [x] Message #1: no fresh deposit tx this run (`transaction: ""`) — this run reused a channel already
  open from earlier testing (persisted in Deno's `localStorage`), not a brand-new one.
  `chargedCumulativeAmount` started at `"2840"` (= 2× the per-message price) rather than `"1420"`,
  confirming the channel already carried state from a prior session.
- [x] `chargedCumulativeAmount` progressed correctly across all three messages: `2840 → 4260 → 5680`
  — a consistent `+1420` per message, matching `USDC_PRICE_PER_MESSAGE`
  (`convertTokensToUsdcCost(LLM_ESTIMATED_TOKENS_PER_MESSAGE)`). No `invalid_batch_settlement_evm_cumulative_amount_mismatch`
  anywhere in the run — both the client-side `readContract` fix and the server-side
  `createPaymentRequiredResponse` enrichment fix hold up end-to-end.
- [x] Messages #2/#3: `fetchWithPayment` **did** reuse the channel automatically — no second deposit,
  no extra 402 round-trip visible in the logs beyond the corrective flow. Message #2's settlement
  receipt included a real on-chain transaction hash (`0x0b2e1093…`), confirming an actual claim/settle
  landed on Base Sepolia (`balance` jumped from `"0"` to `"21300"`); message #3 was a pure off-chain
  voucher (`transaction: ""`, `balance` unchanged).
- [x] No `useX402Chat.ts` workaround needed for channel reuse — the default `fetchWithPayment` +
  `WebStorageClientChannelStorage` behavior already does the right thing automatically.
</cell id="cell-10">
